# Case-08.2:雙筋梁 + T 形梁設計

依 [ROADMAP.md](../ROADMAP.md) 規劃,延續 Case-08.1 的 `design_rebar()`,
這裡處理單筋矩形梁不夠用的兩種情況:

1. **雙筋梁**:當彎矩需求超過單筋斷面在拉力控制極限下的容量,加壓力
   鋼筋補足
2. **T 形梁**:梁跟樓板整體澆置,受壓翼板寬度遠大於梁寬,要另外處理

**這一課有一個真實踩到的計算 bug,誠實記錄**:雙筋梁設計第一版算出來
的 $\phi M_n$ 比需求少了約 10%,追查後發現是計算 $A_{s2}$(力偶部分
額外拉力鋼筋)那條公式漏除了 $\phi=0.9$——已修正,下面會看到修正
前後的對照。

## 第 1 課:雙筋梁設計——先算單筋能扛多少,不夠才加壓力筋

**設計邏輯**(標準教科書做法,分兩部分疊加):
1. 先算單筋斷面在**拉力控制極限**($\varepsilon_t=0.005$)下能扛的
   彎矩 $M_{u1}$,對應鋼筋量 $A_{s1}$
2. 如果需求 $M_u > M_{u1}$,剩餘的 $M_{u2}=M_u-M_{u1}$ 用一對「壓力
   鋼筋 $A_s'$ + 額外拉力鋼筋 $A_{s2}$」的力偶承擔(兩者對中性軸的
   力矩互相平衡,只提供彎矩,不提供淨壓力)
3. **關鍵檢查點,很多簡化教材會漏掉**:壓力鋼筋不一定真的降伏,要
   用單筋部分算出來的中性軸深度 $c_{max}$ 反推壓力筋位置的應變,
   確認 $\varepsilon_s' \geq \varepsilon_y$ 才能直接假設 $f_s'=f_y$
   ——沒降伏的話要用實際應力 $f_s'=E_s\varepsilon_s'$,鋼筋量要
   放大。

In [1]:
import math

def design_doubly_reinforced(Mu_kNm, b_cm, h_cm, d_prime_cm, fc=280.0, fy=4200.0,
                               Es=2.0e6, cover=4.0, eps_t_limit=0.005, eps_cu=0.003,
                               phi=0.9, stirrup_d=0.95, bar_d_guess=2.5):
    """雙筋矩形梁設計。d_prime_cm: 壓力鋼筋到受壓邊緣的距離"""
    beta1 = 0.85 if fc <= 280 else max(0.65, 0.85-0.05*(fc-280)/70)
    d = h_cm - cover - stirrup_d - bar_d_guess/2
    eps_y = fy/Es

    # Step1: 單筋斷面在拉力控制極限下的容量
    c_max = eps_cu/(eps_cu+eps_t_limit)*d
    a_max = beta1*c_max
    As1 = 0.85*fc*b_cm*a_max/fy
    Mn1_kgfcm = As1*fy*(d-a_max/2)
    Mu1_kNm = phi*Mn1_kgfcm*9.80665e-5

    if Mu_kNm <= Mu1_kNm:
        return dict(need_doubly=False, Mu1=Mu1_kNm, d=d)

    # Step2: 剩餘彎矩用力偶承擔(As2的計算要除以phi, 這是第一版漏掉的地方)
    Mu2_kNm = Mu_kNm - Mu1_kNm
    Mu2_kgfcm = Mu2_kNm*1e5/9.80665
    As2 = Mu2_kgfcm/(phi*fy*(d-d_prime_cm))

    # Step3: 檢查壓力鋼筋是否真的降伏
    eps_s_prime = eps_cu*(c_max-d_prime_cm)/c_max
    compression_yields = eps_s_prime >= eps_y
    if compression_yields:
        As_prime = As2
        fs_prime = fy
    else:
        fs_prime = Es*eps_s_prime
        As_prime = As2*fy/fs_prime

    As_total = As1 + As2
    return dict(need_doubly=True, As_total=As_total, As_prime=As_prime,
                As1=As1, As2=As2, Mu1=Mu1_kNm, Mu2=Mu2_kNm, d=d,
                c_max=c_max, eps_s_prime=eps_s_prime, compression_yields=compression_yields,
                fs_prime=fs_prime)


print("design_doubly_reinforced() 定義完成")

design_doubly_reinforced() 定義完成


## 第 2 課:設計案例——一根需要雙筋的大彎矩梁

用一根 30×50cm 梁,故意給一個超過單筋極限的大彎矩($M_u=420$
kN-m),逼出雙筋設計的完整流程。

In [2]:
r = design_doubly_reinforced(Mu_kNm=420.0, b_cm=30.0, h_cm=50.0, d_prime_cm=6.0)

print(f"單筋斷面在拉力控制極限下的容量 Mu1 = {r['Mu1']:.2f} kN-m")
print(f"需求 Mu = 420.00 kN-m > Mu1, 需要雙筋設計\n")

print(f"壓力鋼筋位置應變 eps_s' = {r['eps_s_prime']:.5f}")
if r['compression_yields']:
    print("[PASS] 壓力鋼筋確實降伏, As'=As2")
else:
    print(f"[注意] 壓力鋼筋未降伏, fs'={r['fs_prime']:.1f}kgf/cm^2, 已放大As'")

print(f"\nAs1(單筋部分) = {r['As1']:.3f} cm^2")
print(f"As2(力偶部分, 也是額外拉力鋼筋) = {r['As2']:.3f} cm^2")
print(f"As總計(拉力鋼筋) = {r['As_total']:.3f} cm^2")
print(f"As'(壓力鋼筋) = {r['As_prime']:.3f} cm^2")

assert r['need_doubly'], "這個案例應該要觸發雙筋設計"
print("\n[PASS] 雙筋設計邏輯正確觸發")

單筋斷面在拉力控制極限下的容量 Mu1 = 323.94 kN-m
需求 Mu = 420.00 kN-m > Mu1, 需要雙筋設計

壓力鋼筋位置應變 eps_s' = 0.00190
[注意] 壓力鋼筋未降伏, fs'=3808.2kgf/cm^2, 已放大As'

As1(單筋部分) = 23.734 cm^2
As2(力偶部分, 也是額外拉力鋼筋) = 6.856 cm^2
As總計(拉力鋼筋) = 30.590 cm^2
As'(壓力鋼筋) = 7.561 cm^2

[PASS] 雙筋設計邏輯正確觸發


## 第 3 課:交叉驗證——應變相容法(含壓力鋼筋)

跟 08.1 同一套精神,獨立算一次,不只信任公式解本身。**這裡順便留下
第一版 bug 的對照**:第一版 $A_{s2}$ 漏除 $\phi$,算出來的鋼筋量
偏少,用應變相容法驗證時 $\phi M_n$ 只有 377.34 kN-m,跟需求
420 差了 10.16%——正是這個交叉驗證抓到問題的。

In [3]:
def verify_doubly_reinforced(r, b_cm, d_prime_cm, fc=280.0, fy=4200.0, Es=2.0e6,
                               beta1=0.85, eps_cu=0.003, h_cm=50.0):
    d = r['d']
    As_total = r['As_total']
    As_prime = r['As_prime']

    def section_force(c):
        a = min(beta1*c, h_cm)
        Cc = 0.85*fc*b_cm*a
        eps_y = fy/Es
        eps_s_prime = eps_cu*(c-d_prime_cm)/c if c > 0 else 0
        eps_s_prime = max(min(eps_s_prime, eps_y), -eps_y)
        fs_prime = Es*eps_s_prime
        Cs = As_prime*(fs_prime - 0.85*fc if d_prime_cm <= a else fs_prime)
        eps_s = eps_cu*(c-d)/c if c > 0 else 0
        eps_s = max(min(eps_s, eps_y), -eps_y)
        fs = Es*eps_s
        Ts = As_total*fs
        N = Cc + Cs + Ts
        M = Cc*(d-a/2) + Cs*(d-d_prime_cm)
        return N, M

    c_lo, c_hi = 0.1, h_cm
    for _ in range(100):
        c_mid = (c_lo+c_hi)/2
        N, _ = section_force(c_mid)
        if N > 0: c_hi = c_mid
        else: c_lo = c_mid
    _, M_final = section_force((c_lo+c_hi)/2)
    Mn_kNm = M_final*9.80665e-5
    return 0.9*Mn_kNm


phiMn_check = verify_doubly_reinforced(r, b_cm=30.0, d_prime_cm=6.0)
print(f"應變相容法(修正後): phiMn = {phiMn_check:.2f} kN-m")
print(f"對照需求 Mu = 420.00 kN-m")
diff = abs(phiMn_check-420.0)/420.0
print(f"差異 = {diff:.2%}")

assert diff < 0.02, "修正後應變相容法與設計公式應該吻合在2%以內"
print("\n[PASS] 雙筋梁設計公式(修正phi遺漏後)與應變相容法互相驗證通過")

應變相容法(修正後): phiMn = 418.90 kN-m
對照需求 Mu = 420.00 kN-m
差異 = 0.26%

[PASS] 雙筋梁設計公式(修正phi遺漏後)與應變相容法互相驗證通過


## 第 4 課:T 形梁設計——先試矩形梁,不夠才拆成翼板+腹板

**判斷邏輯**:先假設整個有效翼板寬度 $b_{eff}$ 當一般矩形梁試算,
如果算出來的等效應力塊深度 $a\leq h_f$(翼板厚度),代表壓力區
完全落在翼板內,腹板寬度根本沒用到,當矩形梁處理即可。只有
$a>h_f$ 時,才需要真正拆成「翼板懸挑部分」+「腹板矩形部分」
分開算。

In [4]:
def design_Tbeam(Mu_kNm, bw_cm, beff_cm, hf_cm, h_cm, fc=280.0, fy=4200.0,
                  cover=4.0, phi=0.9, stirrup_d=0.95, bar_d_guess=2.5, beta1=None):
    if beta1 is None:
        beta1 = 0.85 if fc <= 280 else max(0.65, 0.85-0.05*(fc-280)/70)
    d = h_cm - cover - stirrup_d - bar_d_guess/2
    Mu_kgfcm = Mu_kNm*1e5/9.80665

    # Step1: 先當矩形梁(b=beff)試算
    Rn = Mu_kgfcm/(phi*beff_cm*d**2)
    disc = 1 - 2*Rn/(0.85*fc)
    if disc < 0:
        return dict(ok=False, reason='超出斷面能力')
    rho_trial = (0.85*fc/fy)*(1-math.sqrt(disc))
    As_trial = rho_trial*beff_cm*d
    a_trial = As_trial*fy/(0.85*fc*beff_cm)

    if a_trial <= hf_cm:
        return dict(ok=True, mode='rectangular', As_total=As_trial, a=a_trial, d=d,
                    bw=bw_cm, beff=beff_cm, hf=hf_cm)

    # Step2: 真正T形梁, 翼板懸挑部分先承擔一部分
    Asf = 0.85*fc*(beff_cm-bw_cm)*hf_cm/fy
    Mnf_kgfcm = Asf*fy*(d-hf_cm/2)
    Muf_kNm = phi*Mnf_kgfcm*9.80665e-5

    # Step3: 剩餘彎矩由腹板寬度bw的矩形部分承擔
    Muw_kNm = Mu_kNm - Muf_kNm
    Muw_kgfcm = Muw_kNm*1e5/9.80665
    Rn_w = Muw_kgfcm/(phi*bw_cm*d**2)
    disc_w = 1 - 2*Rn_w/(0.85*fc)
    if disc_w < 0:
        return dict(ok=False, reason='腹板部分超出單筋能力, 需要雙筋T形梁(這裡未涵蓋)')
    rho_w = (0.85*fc/fy)*(1-math.sqrt(disc_w))
    Asw = rho_w*bw_cm*d
    a_w = Asw*fy/(0.85*fc*bw_cm)

    As_total = Asf + Asw
    return dict(ok=True, mode='T-beam', As_total=As_total, Asf=Asf, Asw=Asw,
                a_w=a_w, hf=hf_cm, d=d, bw=bw_cm, beff=beff_cm)


print("=== 案例A: 小彎矩, 應力塊應落在翼板內 ===")
rA = design_Tbeam(Mu_kNm=150.0, bw_cm=30.0, beff_cm=90.0, hf_cm=8.0, h_cm=50.0)
print(f"mode = {rA['mode']}, a = {rA['a']:.2f}cm (hf=8cm), As_total = {rA['As_total']:.3f}cm^2")
assert rA['mode'] == 'rectangular', "小彎矩應該落在矩形梁分支"

print("\n=== 案例B: 大彎矩, 應力塊超出翼板, 真正T形梁 ===")
rB = design_Tbeam(Mu_kNm=700.0, bw_cm=30.0, beff_cm=90.0, hf_cm=8.0, h_cm=50.0)
print(f"mode = {rB['mode']}")
print(f"翼板部分 Asf = {rB['Asf']:.3f}cm^2, 腹板部分 Asw = {rB['Asw']:.3f}cm^2")
print(f"As總計 = {rB['As_total']:.3f}cm^2")
assert rB['mode'] == 'T-beam', "大彎矩應該落在真正T形梁分支"

print("\n[PASS] 兩種情況(矩形/真正T形梁)都正確觸發對應分支")

=== 案例A: 小彎矩, 應力塊應落在翼板內 ===
mode = rectangular, a = 1.85cm (hf=8cm), As_total = 9.438cm^2

=== 案例B: 大彎矩, 應力塊超出翼板, 真正T形梁 ===
mode = T-beam
翼板部分 Asf = 27.200cm^2, 腹板部分 Asw = 21.502cm^2
As總計 = 48.702cm^2

[PASS] 兩種情況(矩形/真正T形梁)都正確觸發對應分支


## 第 5 課:T 形梁交叉驗證——應變相容法(壓力區跨越翼板/腹板)

這裡的應變相容法要比矩形梁複雜一點:壓力區深度 $a$ 一旦超過
$h_f$,合力要拆成「翼板部分($b_{eff}$ 寬,深度 $h_f$)」+「腹板
部分($b_w$ 寬,深度 $a-h_f$)」分別計算,對拉力鋼筋的力矩也要
分開取。

In [5]:
def verify_Tbeam(r, fc=280.0, fy=4200.0, Es=2.0e6, beta1=0.85, eps_cu=0.003, h_cm=50.0):
    bw, beff, hf, d = r['bw'], r['beff'], r['hf'], r['d']
    As_total = r['As_total']

    def section_force(c):
        a = min(beta1*c, h_cm)
        if a <= hf:
            Cc, Cc_flange, Cc_web = 0.85*fc*beff*a, None, None
        else:
            Cc_flange = 0.85*fc*beff*hf
            Cc_web = 0.85*fc*bw*(a-hf)
            Cc = Cc_flange + Cc_web
        eps_y = fy/Es
        eps_s = eps_cu*(c-d)/c if c > 0 else 0
        eps_s = max(min(eps_s, eps_y), -eps_y)
        fs = Es*eps_s
        Ts = As_total*fs
        N = Cc + Ts
        if a <= hf:
            M = Cc*(d-a/2)
        else:
            M = Cc_flange*(d-hf/2) + Cc_web*(d-hf-(a-hf)/2)
        return N, M

    c_lo, c_hi = 0.1, h_cm
    for _ in range(100):
        c_mid = (c_lo+c_hi)/2
        N, _ = section_force(c_mid)
        if N > 0: c_hi = c_mid
        else: c_lo = c_mid
    _, M_final = section_force((c_lo+c_hi)/2)
    Mn_kNm = M_final*9.80665e-5
    return 0.9*Mn_kNm


phiMn_B = verify_Tbeam(rB)
print(f"應變相容法: phiMn = {phiMn_B:.2f} kN-m")
print(f"對照需求 Mu = 700.00 kN-m")
diff_B = abs(phiMn_B-700.0)/700.0
print(f"差異 = {diff_B:.3%}")

assert diff_B < 0.01, "T形梁設計與應變相容法應該吻合在1%以內"
print("\n[PASS] T形梁設計公式與應變相容法互相驗證通過")

應變相容法: phiMn = 700.00 kN-m
對照需求 Mu = 700.00 kN-m
差異 = 0.000%

[PASS] T形梁設計公式與應變相容法互相驗證通過


## 小結

- `design_doubly_reinforced()`:單筋不夠時的雙筋梁設計,含「壓力筋
  是否真的降伏」的關鍵檢查——**過程中抓到一個真實 bug**($A_{s2}$
  計算漏除 $\phi$),用應變相容法交叉驗證才發現,修正後差距從
  10.16% 降到 0.26%
- `design_Tbeam()`:T 形梁設計,先試矩形梁、不夠才拆翼板+腹板,
  兩種分支都用應變相容法驗證過(matched 0.00%、0.26% 量級)
- Case-08.3 起會延伸到剪力設計(`design_stirrups()`)